# Baseline Models for Regime Detection

This notebook keeps the comparison deliberately focused: a probabilistic clustering baseline, a distance-based clustering baseline, and an anomaly detector for stress periods. The goal is to benchmark the from-scratch HMM against simple alternatives without diluting the project narrative.

### Market Regime Detection · Probabilistic ML Project


## 0. Imports & Data

In [8]:
%pip install -q scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_regime_stats,
                               REGIME_COLOURS, _DARK_LAYOUT)
from utils.metrics    import regime_statistics, regime_purity

print("All imports OK")

All imports OK


In [14]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

# Feature matrix for multi-variate clustering
FEATURE_COLS = ["Returns", "AbsReturn", "RVol_20d", "RVol_60d", "Momentum_5d"]
X_feat_train = df_train[FEATURE_COLS].values
X_feat_test  = df_test[FEATURE_COLS].values
X_1d_train   = df_train["Returns"].values
X_1d_test    = df_test["Returns"].values

scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_feat_train)
X_scaled_test  = scaler.transform(X_feat_test)

# Reference: observable vol labels for purity computation
vol_labels_train = df_train["VolRegime"].values
vol_labels_test  = df_test["VolRegime"].values

print(f"Train: {len(df_train)}  Test: {len(df_test)}")
print(f"Features: {FEATURE_COLS}")

Train: 364  Test: 92
Features: ['Returns', 'AbsReturn', 'RVol_20d', 'RVol_60d', 'Momentum_5d']


## 1. Gaussian Mixture Model (GMM)

The GMM models the return distribution as a weighted sum of Gaussians — **without** the Markov temporal structure:
$$f(r) = \sum_{j=1}^{K} \pi_j \cdot \mathcal{N}(r;\, \mu_j,\, \sigma_j^2)$$

Fitted via EM (same algorithm as Baum-Welch, but no transition matrix).

**Key difference from HMM:** state transitions are IID — the model has no memory of the previous regime.

In [15]:
# Fit GMM with K=2,3,4,5 and select via BIC
gmm_results = []
for k in range(2, 6):
    g = GaussianMixture(n_components=k, covariance_type="full",
                        random_state=42, n_init=5, max_iter=300)
    g.fit(X_1d_train.reshape(-1, 1))
    gmm_results.append({"K": k, "BIC": g.bic(X_1d_train.reshape(-1,1)),
                         "AIC": g.aic(X_1d_train.reshape(-1,1)), "model": g})

best_k = min(gmm_results, key=lambda r: r["BIC"])["K"]
print(f"BIC-optimal K = {best_k}")
for r in gmm_results:
    print(f"  K={r['K']}  BIC={r['BIC']:.1f}  AIC={r['AIC']:.1f}")

BIC-optimal K = 3
  K=2  BIC=-2341.3  AIC=-2360.8
  K=3  BIC=-2346.5  AIC=-2377.6
  K=4  BIC=-2330.8  AIC=-2373.7
  K=5  BIC=-2325.7  AIC=-2380.2


In [16]:
gmm3 = GaussianMixture(n_components=3, covariance_type="full",
                       random_state=42, n_init=10, max_iter=300)
gmm3.fit(X_1d_train.reshape(-1, 1))

labels_gmm = gmm3.predict(X_1d_train.reshape(-1, 1))

# Sort by std  (low → high vol)
stds_gmm  = np.sqrt(gmm3.covariances_.ravel())
order_gmm = np.argsort(stds_gmm)
remap_gmm = {order_gmm[k]: k for k in range(3)}
labels_gmm = np.array([remap_gmm[l] for l in labels_gmm])

df_gmm = df_train.copy()
df_gmm["GMM"] = labels_gmm

print("GMM means  :", gmm3.means_.ravel()[order_gmm].round(6))
print("GMM stds   :", stds_gmm[order_gmm].round(6))
print("GMM weights:", gmm3.weights_[order_gmm].round(4))

GMM means  : [ 0.099863  0.001993 -0.006248]
GMM stds   : [0.001    0.006358 0.017412]
GMM weights: [0.0027 0.8038 0.1935]


In [17]:
fig = plot_price_with_regimes(df_gmm, "GMM", title="SPY — GMM Regimes")
fig.show()

fig = plot_regime_distributions(df_gmm, "GMM", title="GMM — Regime Return Distributions")
fig.show()

In [18]:
# GMM soft probabilities — no Markov structure
proba_gmm = gmm3.predict_proba(X_1d_train.reshape(-1,1))[:, order_gmm]

fig = go.Figure()
state_names_3 = ["Low-Vol", "Med-Vol", "High-Vol"]
colours_3 = ["rgba(99,200,180,0.7)", "rgba(255,195,55,0.7)", "rgba(240,80,80,0.7)"]
for j in range(3):
    fig.add_trace(go.Scatter(
        x=df_train.index, y=proba_gmm[:, j],
        name=state_names_3[j], stackgroup="one",
        line=dict(color=colours_3[j]),
        fillcolor=colours_3[j],
    ))
fig.update_layout(title="GMM — Soft State Probabilities (no temporal smoothing)",
                  yaxis_title="P(state | return)", height=350, **_DARK_LAYOUT)
fig.show()

print("Purity vs vol labels:", round(regime_purity(labels_gmm, vol_labels_train), 3))

Purity vs vol labels: 0.42


## 2. K-Means Clustering

Hard cluster assignment on scaled feature space (returns, volatility, momentum).

K-Means minimises:
$$\sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2$$

No emission or transition structure is purely geometric.

In [19]:
# Elbow method + silhouette
inertias, silhouettes = [], []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled_train)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled_train, labels_k))
    print(f"K={k}  inertia={km.inertia_:.1f}  silhouette={silhouettes[-1]:.4f}")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Elbow Method", "Silhouette Score"))
fig.add_trace(go.Scatter(x=list(range(2,7)), y=inertias, mode="lines+markers",
    line=dict(color="rgba(99,200,180,0.9)")), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(2,7)), y=silhouettes, mode="lines+markers",
    line=dict(color="rgba(255,195,55,0.9)")), row=1, col=2)
fig.update_layout(height=380, title="K-Means Model Selection", **_DARK_LAYOUT)
fig.show()

K=2  inertia=1352.9  silhouette=0.4911
K=3  inertia=1064.1  silhouette=0.4845
K=4  inertia=872.4  silhouette=0.4842
K=5  inertia=710.1  silhouette=0.4639
K=6  inertia=605.5  silhouette=0.2731


In [20]:
km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_km = km3.fit_predict(X_scaled_train)

# Sort by average return volatility
stds_km  = [X_1d_train[labels_km == k].std() for k in range(3)]
order_km = np.argsort(stds_km)
remap_km = {order_km[k]: k for k in range(3)}
labels_km = np.array([remap_km[l] for l in labels_km])

df_km = df_train.copy()
df_km["KMeans"] = labels_km

fig = plot_price_with_regimes(df_km, "KMeans", title="SPY — K-Means Regimes")
fig.show()

fig = plot_regime_distributions(df_km, "KMeans", title="K-Means — Regime Return Distributions")
fig.show()

print("Purity vs vol labels:", round(regime_purity(labels_km, vol_labels_train), 3))

Purity vs vol labels: 0.462


## 3. Isolation Forest : Stress Period Detection

Isolation Forest assigns **anomaly scores** to each day.
Days with anomaly score < −0.1 are labelled as "stress regime".

In [23]:
iso = IsolationForest(n_estimators=200, contamination=0.08, random_state=42)
iso.fit(X_scaled_train)
anomaly_scores = iso.decision_function(X_scaled_train)
anomaly_labels = (anomaly_scores < -0.05).astype(int)   # 1 = stress

df_iso = df_train.copy()
df_iso["Stress"] = anomaly_labels

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Anomaly Score", "SPY Price"),
                    row_heights=[0.4, 0.6])

fig.add_trace(go.Scatter(x=df_train.index, y=anomaly_scores,
    name="Anomaly Score", line=dict(color="rgba(255,195,55,0.8)", width=0.8)), row=1, col=1)
fig.add_hline(y=-0.05, line_dash="dash", line_color="rgba(240,80,80,0.8)", row=1, col=1)

fig.add_trace(go.Scatter(x=df_train.index, y=df_train["Close"],
    name="SPY Close", line=dict(color="rgba(150,180,255,0.9)", width=1.2)), row=2, col=1)

for t in df_iso.index[df_iso["Stress"] == 1]:
    fig.add_vrect(x0=t, x1=t, fillcolor="rgba(240,80,80,0.15)", line_width=0, row=2, col=1)

fig.update_layout(title="Isolation Forest — Stress Period Detection",
                  height=550, **_DARK_LAYOUT)
fig.show()

print(f"Stress days: {anomaly_labels.sum()} ({anomaly_labels.mean()*100:.1f}% of sample)")
stress_rets = X_1d_train[anomaly_labels == 1]
normal_rets = X_1d_train[anomaly_labels == 0]
print(f"Stress mean/vol: {stress_rets.mean()*252:.2%} / {stress_rets.std()*np.sqrt(252):.2%}")
print(f"Normal mean/vol: {normal_rets.mean()*252:.2%} / {normal_rets.std()*np.sqrt(252):.2%}")

Stress days: 16 (4.4% of sample)
Stress mean/vol: -71.75% / 59.95%
Normal mean/vol: 20.90% / 13.05%


## 4. Key Takeaways

**GMM**
- Clean probabilistic clustering baseline with soft assignments.
- Useful for checking whether return distributions naturally separate into regimes.
- Limitation: treats each day independently and ignores temporal persistence.

**K-Means**
- Simple, fast baseline on engineered features such as returns, realized volatility, and momentum.
- Useful as a sanity check for whether regimes are separable in feature space.
- Limitation: hard boundaries, sensitive to scaling, and no uncertainty estimates.

**Isolation Forest**
- Best framed as a stress-event detector rather than a full regime model.
- Useful for identifying unusual market periods that may correspond to crisis behavior.
- Limitation: binary anomaly output, so it does not provide a persistent multi-state regime structure.

**Why the HMM remains the main model**
- The HMM explicitly models temporal transitions between latent states.
- It provides soft state probabilities, Viterbi paths, and interpretable transition dynamics.
- These baselines are kept to make the comparison grounded without diluting the project narrative.
